<a href="https://colab.research.google.com/github/austinmallie/ADS599_Capstone/blob/main/Feature_Engineering/Hospital_Cost_Report_Feature_Engineered.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering

In [1]:
from google.colab import userdata
import os

token    = userdata.get('GitHub')
owner    = "austinmallie"
repo     = "ADS599_Capstone"
repo_url = f"https://{token}@github.com/{owner}/{repo}.git"

# Only clone if the folder doesn't exist already
if not os.path.exists(repo):
    !git clone {repo_url}
else:
    print("Repo already cloned. Pulling latest changes...")
    %cd {repo}
    !git pull

%cd /content/{repo}


Cloning into 'ADS599_Capstone'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 183 (delta 34), reused 15 (delta 15), pack-reused 139 (from 1)
Receiving objects: 100% (183/183), 24.37 MiB | 20.64 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/ADS599_Capstone


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import warnings, re, os
warnings.filterwarnings('ignore')

In [3]:
FILE_MAP = {
    2014: 'Hospital_Provider_Cost_Report_2014.csv',
    2015: 'Hospital_Provider_Cost_Report_2015.csv',
    2016: 'Hospital_Provider_Cost_Report_2016.csv',
    2017: 'Hospital_Provider_Cost_Report_2017.csv',
    2018: 'Hospital_Provider_Cost_Report_2018.csv',
    2019: 'Hospital_Provider_Cost_Report_2019.csv',
    2021: 'Hospital_Provider_Cost_Report_2021.csv',
    2022: 'Hospital_Provider_Cost_Report_2022.csv',
    2023: 'CostReport_2023_Final.csv',
}

frames = []
for year, fname in FILE_MAP.items():
    path = os.path.join("Data-Folder/Cost-Report-Data", fname)
    tmp  = pd.read_csv(path, low_memory=False)
    tmp['report_year'] = year
    frames.append(tmp)
    print(f"  {year}: {len(tmp):,} rows  |  {tmp.shape[1]} cols")

print(f"\n Total files loaded: {len(frames)}")


  2014: 6,250 rows  |  118 cols
  2015: 6,257 rows  |  118 cols
  2016: 6,211 rows  |  118 cols
  2017: 6,174 rows  |  118 cols
  2018: 6,160 rows  |  118 cols
  2019: 6,121 rows  |  118 cols
  2021: 6,053 rows  |  118 cols
  2022: 6,064 rows  |  118 cols
  2023: 6,103 rows  |  118 cols

 Total files loaded: 9


In [4]:

all_cols   = [set(f.columns) for f in frames]
base_cols  = set.intersection(*all_cols)
extra_cols = set.union(*all_cols) - base_cols

print(f"Columns present in ALL years  : {len(base_cols)}")
print(f"Columns present in SOME years : {len(extra_cols)}")
if extra_cols:
    print("  Extra columns:", extra_cols)

Columns present in ALL years  : 118
Columns present in SOME years : 0


In [5]:
# Standardize Column Names: strip whitespace, lowercase, replace special chars
def clean_col(c):
    c = c.strip()
    c = re.sub(r'[^A-Za-z0-9]+', '_', c)
    return c.lower().strip('_')

for i, f in enumerate(frames):
    frames[i].columns = [clean_col(c) for c in f.columns]

#  Stack all years
df = pd.concat(frames, ignore_index=True, sort=False)

print(f"Combined shape : {df.shape}")
print(f"Years present  : {sorted(df['report_year'].unique())}")
print(f"Memory usage   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Combined shape : (55393, 118)
Years present  : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023)]
Memory usage   : 80.1 MB


In [6]:
# Standardise date columns
for col in ['fiscal_year_begin_date', 'fiscal_year_end_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Rename key columns for readability
rename = {
    'provider_ccn'                              : 'ccn',
    'state_code'                                : 'state',
    'rural_versus_urban'                        : 'rural_urban',
    'type_of_control'                           : 'control_type',
    'number_of_beds'                            : 'beds',
    'total_costs'                               : 'total_costs',
    'net_patient_revenue'                       : 'net_patient_revenue',
    'net_income'                                : 'net_income',
    'total_patient_revenue'                     : 'total_patient_revenue',
    'less_total_operating_expense'              : 'total_operating_expense',
    'total_salaries_from_worksheet_a'           : 'total_salaries',
    'total_discharges_v_xviii_xix_unknown'      : 'total_discharges',
    'total_days_v_xviii_xix_unknown'            : 'total_days',
    'inpatient_revenue'                         : 'inpatient_revenue',
    'outpatient_revenue'                        : 'outpatient_revenue',
    'cost_of_charity_care'                      : 'charity_care_cost',
    'total_bad_debt_expense'                    : 'bad_debt_expense',
    'total_assets'                              : 'total_assets',
    'total_liabilities'                         : 'total_liabilities',
    'total_fund_balances'                       : 'total_fund_balances',
    'fte_employees_on_payroll'                  : 'fte_employees',
    'contract_labor_direct_patient_care'        : 'contract_labor',
    'disproportionate_share_adjustment'         : 'dsh_adjustment',
    'allowable_dsh_percentage'                  : 'dsh_pct',
    'net_revenue_from_medicaid'                 : 'medicaid_net_revenue',
    'cost_to_charge_ratio'                      : 'cost_to_charge_ratio',
    'depreciation_cost'                         : 'depreciation_cost',
    'overhead_non_salary_costs'                 : 'overhead_costs',
    'medicare_cbsa_number'                      : 'cbsa',
}
df.rename(columns={k: v for k, v in rename.items() if k in df.columns}, inplace=True)

print(f" Schema reconciled. Final columns: {df.shape[1]}")
df.head(2)

 Schema reconciled. Final columns: 118


,rpt_rec_num,ccn,hospital_name,street_address,city,state,zip_code,county,cbsa,rural_urban,...,total_other_income,total_income,total_other_expenses,net_income,cost_to_charge_ratio,medicaid_net_revenue,medicaid_charges,net_revenue_from_stand_alone_chip,stand_alone_chip_charges,report_year
0,549185,104075,NORTH TAMPA BEHAVIORAL SYSTEM,29910 SR 56,WESLEY CHAPEL,FL,33543,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014
1,550904,514011,HIGHLAND CLARKSBUGS,3 HOSPITAL PLAZA,CLARKSBURG,WV,26301,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014


In [7]:
# Overall missing-value summary
total   = len(df)
miss    = df.isnull().sum()
miss_pct = (miss / total * 100).round(2)
miss_df = (pd.DataFrame({'missing_count': miss, 'missing_pct': miss_pct})
             .query('missing_count > 0')
             .sort_values('missing_pct', ascending=False))

print(f"Total rows  : {total:,}")
print(f"Total cols  : {df.shape[1]}")
print(f"Cols with nulls: {len(miss_df)}\n")
print(miss_df.head(30).to_string())

Total rows  : 55,393
Total cols  : 118
Cols with nulls: 108

                                                   missing_count  missing_pct
drg_amounts_other_than_outlier_payments                    55393       100.00
hospital_total_days_title_v_for_adults_peds                54008        97.50
hospital_total_discharges_title_v_for_adults_peds          53945        97.39
total_discharges_title_v                                   53945        97.39
total_days_title_v                                         53873        97.26
notes_receivable                                           52814        95.34
unsecured_loans                                            52750        95.23
wage_related_costs_rhc_fqhc                                51853        93.61
wage_related_costs_for_part_a_teaching_physicians          51699        93.33
health_information_technology_designated_assets            50438        91.05
wage_related_costs_for_interns_and_residents               48274        87.15
mor

In [8]:
# threshold
MISSING_THRESHOLD = 0.50

# calculate missing percentage
missing_pct = df.isnull().mean()

# identify columns to drop
cols_to_drop = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()

print(f"Columns removed (>50% missing): {len(cols_to_drop)}")
print(cols_to_drop[:10])  # preview first 10

# drop them
df_clean = df.drop(columns=cols_to_drop)

print("\nNew dataset shape:")
print(df_clean.shape)

Columns removed (>50% missing): 31
['number_of_interns_and_residents_fte', 'total_days_title_v', 'total_discharges_title_v', 'hospital_total_days_title_v_for_adults_peds', 'hospital_total_discharges_title_v_for_adults_peds', 'wage_related_costs_rhc_fqhc', 'wage_related_costs_for_part_a_teaching_physicians', 'wage_related_costs_for_interns_and_residents', 'temporary_investments', 'notes_receivable']

New dataset shape:
(55393, 87)


In [9]:
OUTPUT_PATH = "/content/ADS599_Capstone/Feature_Engineering/hospital_cost_report_feature_engineered.csv"

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Final shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")

Saved to: /content/ADS599_Capstone/Feature_Engineering/hospital_cost_report_feature_engineered.csv
Final shape: 55,393 rows x 87 columns
